# Commandes shell pour le déploiement

Ce notebook regroupe les commandes utiles pour compiler un script Python en exécutable Windows, créer un raccourci de démarrage automatique, contrôler l’installation et nettoyer un ancien déploiement.

Les chemins et noms ci-dessous sont génériques. Remplacer les variables par les valeurs adaptées au poste avant exécution.


## 1. Variables de travail PowerShell

Définir les chemins une seule fois limite les erreurs de copier-coller. Le nom du script, le nom de l’exécutable et les dossiers peuvent être adaptés à chaque poste.


In [ ]:
$ProjectDir = "C:\chemin\vers\le\projet"
$ScriptName = "nom_du_script.py"
$ExeName    = "nom_de_l_executable"
$DistDir    = Join-Path $ProjectDir "dist"
$ExePath    = Join-Path $DistDir "$ExeName.exe"


## 2. Générer un exécutable avec PyInstaller

Cette commande produit un fichier `.exe` autonome dans le dossier `dist`. L’option `--noconsole` masque la fenêtre console, ce qui convient à un programme lancé automatiquement en arrière-plan.


In [ ]:
cd $ProjectDir
pyinstaller --onefile --noconsole --name $ExeName $ScriptName


## 3. Créer un raccourci au démarrage Windows

Le dossier `Startup` lance automatiquement les raccourcis qu’il contient à l’ouverture de session. Le raccourci pointe vers l’exécutable généré et définit son dossier de travail.


In [ ]:
$StartupDir = [System.Environment]::GetFolderPath("Startup")
$ShortcutPath = Join-Path $StartupDir "$ExeName.lnk"

$Shell = New-Object -ComObject WScript.Shell
$Shortcut = $Shell.CreateShortcut($ShortcutPath)
$Shortcut.TargetPath = $ExePath
$Shortcut.WorkingDirectory = $DistDir
$Shortcut.Save()


## 4. Vérifier le raccourci de démarrage

Cette vérification permet de confirmer que le raccourci a bien été créé dans le dossier utilisé par Windows pour le démarrage automatique.


In [ ]:
Get-ChildItem ([System.Environment]::GetFolderPath("Startup"))


## 5. Arrêter un processus en cours

Avant de remplacer ou supprimer un exécutable, arrêter le processus correspondant. Cette commande ignore l’erreur si le processus n’est pas actif.


In [ ]:
Stop-Process -Name $ExeName -Force -ErrorAction SilentlyContinue


## 6. Supprimer le raccourci de démarrage

Cette étape désactive le lancement automatique sans supprimer le programme lui-même.


In [ ]:
$StartupDir = [System.Environment]::GetFolderPath("Startup")
$ShortcutPath = Join-Path $StartupDir "$ExeName.lnk"
Remove-Item $ShortcutPath -Force -ErrorAction SilentlyContinue


## 7. Nettoyer les fichiers générés

PyInstaller crée généralement les dossiers `build`, `dist` et un fichier `.spec`. Les supprimer permet de repartir d’un état propre avant une nouvelle compilation.


In [ ]:
Remove-Item (Join-Path $ProjectDir "build") -Recurse -Force -ErrorAction SilentlyContinue
Remove-Item (Join-Path $ProjectDir "dist")  -Recurse -Force -ErrorAction SilentlyContinue
Remove-Item (Join-Path $ProjectDir "$ExeName.spec") -Force -ErrorAction SilentlyContinue


## 8. Variante CMD pour diagnostic rapide

Ces commandes CMD sont utiles lorsqu’un accès PowerShell n’est pas disponible. Elles permettent de lister le dossier de démarrage et d’arrêter un processus.


In [ ]:
dir "%APPDATA%\Microsoft\Windows\Start Menu\Programs\Startup"
taskkill /f /im nom_de_l_executable.exe /t
